In [0]:
from pyspark.sql.functions import col, concat, lit, when
import time

print("=== CELL 1: Generating Fact and Dimension Tables ===")

# 1. The Skewed Fact Table (100 Million rows: 90M GUEST, 10M unique users)
clickstream_df = spark.range(0, 100000000).withColumn(
    "user_id",
    when(col("id") < 90000000, lit("GUEST"))
    .otherwise(concat(lit("user_"), col("id")))
).withColumn("action", lit("click_homepage"))

# 2. The Dimension Table (10 Million rows)
user_profiles_df = spark.range(90000000, 100000000).withColumn(
    "user_id",
    concat(lit("user_"), col("id"))
).withColumn("user_level", lit("Silver")).drop("id") # Drops 'id' to maintain exactly 2 columns

# 3. Create the matching 2-column GUEST profile
guest_profile_df = spark.createDataFrame([("GUEST", "No Level")], ["user_id", "user_level"])

# Union the datasets together seamlessly
user_profiles_df = user_profiles_df.union(guest_profile_df)

print("✅ Success! Both dataframes are perfectly aligned and ready.")

In [0]:
import time

print("=== CELL 2: Running Unsalted Shuffle Join... ===")
start_time = time.time()

# Force a Shuffle Hash Join to simulate both tables being huge
unsalted_join_df = clickstream_df.join(
    user_profiles_df.hint("shuffle_hash"), 
    on="user_id", 
    how="inner"
)

# Trigger the action to force network shuffling
total_rows = unsalted_join_df.count()

unsalted_duration = time.time() - start_time
print(f"Total Rows Joined: {total_rows}")
print(f"⏰ Unsalted Join Runtime: {unsalted_duration:.2f} seconds")

In [0]:
from pyspark.sql.functions import col, concat, lit, floor, rand, when, array, explode
import time

print("=== CELL 3: Running Targeted Salted Join... ===")
start_time = time.time()

SALT_FACTOR = 5 # Chop the 90 million guest rows into 5 even network streams

# Step 1: Salt ONLY the "GUEST" rows in the Fact table
df_salted_fact = clickstream_df.withColumn(
    "salted_user_id",
    when(col("user_id") == "GUEST", 
         concat(col("user_id"), lit("_"), floor(rand() * SALT_FACTOR)))
    .otherwise(col("user_id"))
)

# Step 2: Replicate ONLY the "GUEST" row 5 times in the Dimension table
# FIXED: cast str(i) so it creates an array of strings: ["0", "1", "2", "3", "4"]
salt_array = array([lit(str(i)) for i in range(SALT_FACTOR)])

df_salted_dim = user_profiles_df.withColumn(
    "salt_box", 
    when(col("user_id") == "GUEST", salt_array).otherwise(array(lit("NO_SALT")))
).withColumn("exploded_salt", explode("salt_box")) \
 .withColumn("salted_user_id", 
             when(col("exploded_salt") == "NO_SALT", col("user_id"))
             .otherwise(concat(col("user_id"), lit("_"), col("exploded_salt")))) \
 .drop("salt_box", "exploded_salt")

# Step 3: Run the shuffle join on our balanced, salted keys
salted_join_df = df_salted_fact.join(
    df_salted_dim.hint("shuffle_hash"), 
    on="salted_user_id", 
    how="inner"
)

# Trigger the action
total_salted_rows = salted_join_df.count()

salted_duration = time.time() - start_time
print(f"Total Rows Joined: {total_salted_rows}")
print(f"⏰ Salted Join Runtime: {salted_duration:.2f} seconds")

In [0]:
from pyspark.sql.functions import col

# Grab a random 1% sample of the data and check the top 10 most frequent keys
skew_check_df = clickstream_df.sample(withReplacement=False, fraction=0.01) \
                              .groupBy("user_id") \
                              .count() \
                              .orderBy(col("count").desc())

display(skew_check_df.limit(10))

In [0]:
%sql
-- Run this once to populate or refresh the catalog statistics
ANALYZE TABLE clickstream_table COMPUTE STATISTICS FOR COLUMNS user_id;

-- Describe the details to look at distinct counts and null profiles
DESCRIBE DETAIL clickstream_table;